In [14]:
import sys
sys.path.insert(0, '/user/bnc2119/drd')
import numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib as mpl
import pandas as pd
from src.drd import DRD, AutoEncoder# stored in base
from matplotlib.ticker import ScalarFormatter
from utils import load_and_split, get_teacher_embeddings
import pickle
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
# build the styled table
styles = [
    # collapse borders so they’re shared
    {"selector": "table", "props": [("border-collapse", "collapse")]},

    # default 1px light‐gray grid
    {"selector": "th, td", 
     "props": [("border", "1px solid #ccc"), ("padding", "5px")]},

    # thick line below each teacher group
    {"selector": "tbody th.row_heading.level0",
     "props": [("border-bottom", "3px solid #444")]}
]

def get_cols(agg):
    distill_cols = agg.columns[(agg.columns.get_level_values(0) == "distill_loss") &
        (agg.columns.get_level_values(1) == "mean")]
    recon_cols   = agg.columns[(agg.columns.get_level_values(0) == "recon_loss") &
        (agg.columns.get_level_values(1) == "mean")]
    time_cols    = agg.columns[(agg.columns.get_level_values(0) == "time_total_s") &
        (agg.columns.get_level_values(1) == "mean")]
    return distill_cols, recon_cols, time_cols

In [ ]:
analysis_umap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/wine_umap_neigh15.csv')
analysis_umap['teacher'] = "umap_neigh15"
analysis_pca = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/wine_pca.csv')
analysis_pca['teacher'] = "pca"
analysis_isomap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/wine_isomap_neigh15.csv')
analysis_isomap['teacher'] = "isomap_neigh15"
analysis = pd.concat([analysis_umap, analysis_pca, analysis_isomap], ignore_index=True)
analysis.fillna('None', inplace=True)
agg = analysis.groupby(['teacher', 'config/activation', 'config/bottleneck_activation', 'config/hidden_dims']).agg({"distill_loss": ['mean', 'std', 'count'], "recon_loss": ['mean', 'std'], "time_total_s": ['mean']})  

distill_cols, recon_cols, time_cols = get_cols(agg)
(
    agg.style
      .background_gradient(
          subset=distill_cols,
          cmap="Blues",
          vmin=agg[distill_cols].min().min(),
          vmax=agg[distill_cols].max().max()
       )
      .background_gradient(
          subset=recon_cols,
          cmap="Oranges",
          vmin=agg[recon_cols].min().min(),
          vmax=agg[recon_cols].max().max()
       )
       .background_gradient(
          subset=time_cols,
          cmap="Greens",
          vmin=agg[time_cols].min().min(),
          vmax=agg[time_cols].max().max()
       )
      .format({
          **{col: "{:.4e}"  for col in distill_cols},    
          **{col: "{:.6f}" for col in recon_cols}        
       })
      .set_caption("Distill & Recon Loss (mean/std/count)")
      .set_table_styles(styles)
)

In [49]:
analysis_umap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/gene_cancer_umap_neigh10.csv')
analysis_umap['teacher'] = "umap_neigh10"
analysis_pca = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/gene_cancer_pca.csv')
analysis_pca['teacher'] = "pca"
analysis_isomap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/gene_cancer_isomap_neigh10.csv')
analysis_isomap['teacher'] = "isomap_neigh10"
analysis = pd.concat([analysis_umap, analysis_pca, analysis_isomap], ignore_index=True)
analysis.fillna('None', inplace=True)
agg = analysis.groupby(['teacher', 'config/activation', 'config/bottleneck_activation', 'config/hidden_dims']).agg({"distill_loss": ['mean', 'std', 'count'], "recon_loss": ['mean', 'std'], "time_total_s": ['mean']})  

distill_cols, recon_cols, time_cols = get_cols(agg)

(
    agg.style
      .background_gradient(
          subset=distill_cols,
          cmap="Blues",
          vmin=agg[distill_cols].min().min(),
          vmax=10
       )
      .background_gradient(
          subset=recon_cols,
          cmap="Oranges",
          vmin=agg[recon_cols].min().min(),
          vmax=agg[recon_cols].max().max()
       )
       .background_gradient(
          subset=time_cols,
          cmap="Greens",
          vmin=agg[time_cols].min().min(),
          vmax=agg[time_cols].max().max()
       )
      .format({
          **{col: "{:.4e}"  for col in distill_cols},    
          **{col: "{:.6f}" for col in recon_cols}        
       })
      .set_caption("Distill & Recon Loss (mean/std/count)")
      .set_table_styles(styles)
)

/tmp/ipykernel_361251/511101279.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'None' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  analysis.fillna('None', inplace=True)


In [50]:
# this experiment marginalizes over all other tuning hyperparameters (lr, eta_min, etc)
# mnist data, pca teacher
analysis_pca = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/mnist_pca.csv')
analysis_pca['teacher'] = "pca"
analysis = pd.concat([analysis_pca], ignore_index=True)
analysis.fillna('None', inplace=True)
agg = analysis.groupby(['teacher', 'config/activation', 'config/bottleneck_activation', 'config/hidden_dims']).agg({"distill_loss": ['mean', 'std', 'count'], "recon_loss": ['mean', 'std'], "time_total_s": ['mean']})  

distill_cols, recon_cols, time_cols = get_cols(agg)

(
    agg.style
      .background_gradient(
          subset=distill_cols,
          cmap="Blues",
          vmin=agg[distill_cols].min().min(),
          vmax=agg[distill_cols].max().max()
       )
      .background_gradient(
          subset=recon_cols,
          cmap="Oranges",
          vmin=agg[recon_cols].min().min(),
          vmax=agg[recon_cols].max().max()
       )
       .background_gradient(
          subset=time_cols,
          cmap="Greens",
          vmin=agg[time_cols].min().min(),
          vmax=agg[time_cols].max().max()
       )
      .format({
          **{col: "{:.4e}"  for col in distill_cols},   
          **{col: "{:.6f}" for col in recon_cols}        
       })
      .set_caption("Distill & Recon Loss (mean/std/count)")
      .set_table_styles(styles)
)

In [53]:
analysis_pca = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/diabetes_pca.csv')
analysis_pca['teacher'] = "pca"
analysis_umap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/diabetes_umap_neigh10.csv')
analysis_umap['teacher'] = "umap_neigh10"
analysis_isomap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/diabetes_isomap_neigh10.csv')
analysis_isomap['teacher'] = "isomap_neigh10"
analysis = pd.concat([analysis_pca, analysis_umap, analysis_isomap], ignore_index=True)
analysis.fillna('None', inplace=True)
agg = analysis.groupby(['teacher', 'config/activation', 'config/bottleneck_activation', 'config/hidden_dims']).agg({"distill_loss": ['mean', 'std', 'count'], "recon_loss": ['mean', 'std'], "time_total_s": ['mean']})  

distill_cols, recon_cols, time_cols = get_cols(agg)

(
    agg.style
      .background_gradient(
          subset=distill_cols,
          cmap="Blues",
          vmin=agg[distill_cols].min().min(),
          vmax=agg[distill_cols].max().max()
       )
      .background_gradient(
          subset=recon_cols,
          cmap="Oranges",
          vmin=agg[recon_cols].min().min(),
          vmax=agg[recon_cols].max().max()
       )
       .background_gradient(
          subset=time_cols,
          cmap="Greens",
          vmin=agg[time_cols].min().min(),
          vmax=agg[time_cols].max().max()
       )
      .format({
          **{col: "{:.4e}"  for col in distill_cols},   
          **{col: "{:.6f}" for col in recon_cols}     
       })
      .set_caption("Distill & Recon Loss (mean/std/count) (not tuned)")
      .set_table_styles(styles)
)

In [52]:
pd.set_option('display.float_format', '{:.6e}'.format)
analysis_pca = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/single_cell_pca.csv')
analysis_pca['teacher'] = "pca"
# analysis_umap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/single_cell_umap_neigh10.csv')
# analysis_umap['teacher'] = "umap_neigh10"
# analysis_isomap = pd.read_csv('/shared/share_mala/irchang/drd/tune_results/single_cell_isomap_neigh10.csv')
# analysis_isomap['teacher'] = "isomap_neigh10"
analysis = pd.concat([analysis_pca], ignore_index=True)
analysis.fillna('None', inplace=True)
agg = analysis.groupby(['teacher', 'config/activation', 'config/bottleneck_activation', 'config/hidden_dims']).agg({"distill_loss": ['mean', 'std', 'count'], "recon_loss": ['mean', 'std'], "time_total_s": ['mean']})  

distill_cols, recon_cols, time_cols = get_cols(agg)

(
    agg.style
      .background_gradient(
          subset=distill_cols,
          cmap="Blues",
          vmin=agg[distill_cols].min().min(),
          vmax=agg[distill_cols].max().max()
       )
      .background_gradient(
          subset=recon_cols,
          cmap="Oranges",
          vmin=agg[recon_cols].min().min(),
          vmax=agg[recon_cols].max().max()
       )
       .background_gradient(
          subset=time_cols,
          cmap="Greens",
          vmin=agg[time_cols].min().min(),
          vmax=agg[time_cols].max().max()
       )
      .format({
          **{col: "{:.4e}"  for col in distill_cols},   
          **{col: "{:.6f}" for col in recon_cols}     
       })
      .set_caption("Distill & Recon Loss (mean/std/count) (not tuned)")
      .set_table_styles(styles)
)